In [1]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026  |  Revisado: 06-06-2026 (v2)
# Uso:     Estatística avançada (MK, KW, Descritiva), P95 e plots (storytelling)
#          para todo o escopo CONAMA 430.
# v2:      - Caminhos via pathlib  - Warnings direcionados  - Plots redesenhados
#            (decluttering, hierarquia tipográfica, rotulagem direta).
#          - Eixo de datas em PORTUGUÊS (formatador independente de locale).
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import pymannkendall as mk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

# Supressão DIRECIONADA: apenas avisos esperados de séries constantes
# (parâmetros 100% <LQ ficam sem variância após a imputação LQ/sqrt2).
_ciw = getattr(stats, 'ConstantInputWarning', None)
if _ciw is not None:
    warnings.filterwarnings("ignore", category=_ciw)
warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Precision loss occurred")

# --- 1. CONFIGURAÇÕES (caminhos centralizados em uma raiz) ---
RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {
    'primary': '#009CA7', 'secondary': '#004851', 'vmp': '#F25A3C',
    'ok': '#52BD8B', 'text': '#404041', 'muted': '#9aa0a6', 'highlight': '#B9F8FF',
}

# Meses em portugues (independe de locale do sistema operacional)
MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def formatar_meses_pt(ax):
    """Eixo X de datas com abreviações de mês em PT-BR (ex.: Mai/25, Out/25)."""
    def _fmt(x, pos=None):
        d = mdates.num2date(x)
        return f"{MESES_PT[d.month - 1]}/{d:%y}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))

# --- ESTILO STORYTELLING (fonte com fallback p/ portabilidade) ---
def configurar_estilo():
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
        'font.size': 11, 'axes.labelsize': 11, 'axes.labelcolor': NESA_COLORS['text'],
        'axes.edgecolor': '#d6d6d6', 'axes.linewidth': 1.0, 'axes.axisbelow': True,
        'axes.grid': True, 'grid.color': '#ececec', 'grid.linewidth': 0.9,
        'xtick.color': NESA_COLORS['text'], 'ytick.color': NESA_COLORS['text'],
        'text.color': NESA_COLORS['text'], 'figure.dpi': 110,
        'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False,
    })

configurar_estilo()

def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    """Hierarquia tipográfica: manchete em negrito + linha de insight + fonte."""
    ax.set_title('')
    ax.text(0.0, 1.13, titulo, transform=ax.transAxes, fontsize=15,
            fontweight='bold', color='#2b2b2b', va='bottom', ha='left')
    if subtitulo:
        ax.text(0.0, 1.04, subtitulo, transform=ax.transAxes, fontsize=10.5,
                color=NESA_COLORS['muted'], va='bottom', ha='left')
    if fonte:
        ax.text(0.0, -0.30, fonte, transform=ax.transAxes, fontsize=8,
                color=NESA_COLORS['muted'], va='top', ha='left')

def limpar_eixos(ax, manter_esquerda=True):
    """Remove cromo desnecessário (spines/ticks/grid vertical)."""
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
    if not manter_esquerda:
        ax.spines['left'].set_visible(False)
    ax.grid(axis='x', visible=False)
    ax.tick_params(length=0)

# --- 2. VMPs CONAMA 430/11 (Art. 16) ---
VMP_CONAMA = {
    'DEMANDA_BIOQUIMICA_DE_OXIGENIO': 120.0,
    'PH': (5.0, 9.0),
    'TEMPERATURA_DO_EFLUENTE': 40.0,
    'MATERIAIS_SEDIMENTAVEIS': 1.0,
    'OLEOS_E_GRAXAS': 50.0,
    'ARSENIO_TOTAL': 0.5,
    'CADMIO_TOTAL': 0.2,
    'CHUMBO_TOTAL': 0.5,
    'CIANETO_TOTAL': 1.0,
    'CROMO_HEXAVALENTE': 0.1,
    'CROMO_TRIVALENTE': 1.0,
    'FERRO_DISSOLVIDO': 15.0,
    'MANGANES_DISSOLVIDO': 1.0,
    'MERCURIO_TOTAL': 0.01,
    'SULFETO_TOTAL': 1.0,
    'BENZENO': 1.2,
    'CLOROFORMIO': 1.0,
    'ETILBENZENO': 1.4,
    'FENOIS_TOTAIS': 0.5,
    'TETRACLORETO_DE_CARBONO': 1.0,
    'TRICLOROETENO': 1.0,
    'TOLUENO': 1.2,
    'XILENO_TOTAL': 1.6,
}

# Unidade exibida por parâmetro (default mg/L); '' = adimensional (pH)
UNIDADES = {'PH': '', 'TEMPERATURA_DO_EFLUENTE': '°C'}

# --- 3. MOTORES ESTATÍSTICOS ---
def analisar_parametro(df, parametro, limite_conama, col_ete):
    """Calcula Descritiva, Auditoria P95 e Tendência de Mann-Kendall."""
    resultados = []
    for ete in df[col_ete].dropna().unique():
        df_ete = df[df[col_ete] == ete].sort_values('DATA')
        serie = df_ete[parametro].dropna()
        if len(serie) < 3:
            continue

        media, desvio = serie.mean(), serie.std()
        minimo, maximo = serie.min(), serie.max()
        p95 = np.percentile(serie, 95)

        if isinstance(limite_conama, tuple):
            vmin, vmax = limite_conama
            amostras_ok = len(serie[(serie >= vmin) & (serie <= vmax)])
            status_auditoria = "ROBUSTO" if (serie.min() >= vmin and p95 <= vmax) else "RISCO"
            limite_str = f"{vmin} a {vmax}"
        else:
            amostras_ok = len(serie[serie <= limite_conama])
            status_auditoria = "ROBUSTO" if p95 <= limite_conama else "RISCO"
            limite_str = str(limite_conama)

        taxa_conformidade = (amostras_ok / len(serie)) * 100

        if desvio > 0 and len(serie) >= 4:
            try:
                mk_res = mk.original_test(serie)
                tendencia, p_val_mk = mk_res.trend, mk_res.p
                if tendencia == 'increasing': tendencia = 'Crescente (Piora)'
                elif tendencia == 'decreasing': tendencia = 'Decrescente (Melhora)'
                else: tendencia = 'Estável'
            except Exception:
                tendencia, p_val_mk = "Erro Cálculo", np.nan
        else:
            tendencia, p_val_mk = "Sem variância/dados", np.nan

        resultados.append({
            'Parâmetro': parametro.replace('_', ' '),
            'ETE': ete, 'N': len(serie),
            'Mínimo': round(minimo, 3), 'Média': round(media, 3),
            'Máximo': round(maximo, 3), 'Desvio Padrão': round(desvio, 3),
            'P95 (Auditoria)': round(p95, 3), 'VMP CONAMA': limite_str,
            'Conformidade (%)': f"{taxa_conformidade:.1f}%",
            'Avaliação': status_auditoria, 'Tendência (MK)': tendencia,
            'p-value (MK)': round(p_val_mk, 4) if pd.notna(p_val_mk) else "N/A",
        })
    return resultados

def teste_kruskal_wallis_geral(df, parametro, col_ete):
    """Verifica se há diferença significativa de desempenho entre as ETEs."""
    grupos, nomes_etes = [], []
    for ete in df[col_ete].dropna().unique():
        serie = df[df[col_ete] == ete][parametro].dropna()
        if len(serie) >= 3 and serie.std() > 0:
            grupos.append(serie)
            nomes_etes.append(ete)
    if len(grupos) > 1:
        stat, p_val = stats.kruskal(*grupos)
        return {
            'Parâmetro': parametro.replace('_', ' '),
            'ETEs Comparadas': " vs ".join(nomes_etes),
            'Estatística (H)': round(stat, 3),
            'p-value (KW)': round(p_val, 4),
            'Diferença Significativa?': "SIM" if p_val < 0.05 else "NÃO",
        }
    return None

def plotar_serie_temporal(df, parametro, ete, col_ete, dir_out):
    """Plot de série temporal redesenhado (storytelling)."""
    df_plot = df[df[col_ete] == ete].sort_values('DATA').dropna(subset=[parametro])
    if len(df_plot) < 2:
        return
    serie = df_plot[parametro]
    limite = VMP_CONAMA[parametro]
    p95 = np.percentile(serie, 95)
    unidade = UNIDADES.get(parametro, 'mg/L')

    fig, ax = plt.subplots(figsize=(11, 5.2))
    limpar_eixos(ax)
    ytrans = ax.get_yaxis_transform()

    # Faixa/linha(s) de limite (laranja = alerta) e zona conforme (verde suave)
    if isinstance(limite, tuple):
        ax.axhspan(limite[0], limite[1], color=NESA_COLORS['ok'], alpha=0.06, zorder=0)
        for y in limite:
            ax.axhline(y, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.4, zorder=1)
            ax.text(1.005, y, f' VMP {y:g}', transform=ytrans, color=NESA_COLORS['vmp'],
                    fontsize=9, va='center', fontweight='bold')
        topo, base = limite[1], limite[0]
    else:
        ax.axhline(limite, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.6, zorder=1)
        ax.text(1.005, limite, f' VMP {limite:g}', transform=ytrans, color=NESA_COLORS['vmp'],
                fontsize=9, va='center', fontweight='bold')
        topo, base = limite, 0.0

    # Linha de dados (cor de marca) com marcadores claros
    ax.plot(df_plot['DATA'], serie, marker='o', markersize=5, linewidth=2.4,
            color=NESA_COLORS['primary'], zorder=3, markerfacecolor='white',
            markeredgecolor=NESA_COLORS['primary'], markeredgewidth=1.6)

    # P95 - linha de auditoria
    ax.axhline(p95, color=NESA_COLORS['secondary'], linestyle=':', linewidth=1.8, zorder=2)
    ax.text(1.005, p95, f' P95 {p95:.2g}', transform=ytrans, color=NESA_COLORS['secondary'],
            fontsize=9, va='center')

    # Rotulagem direta do último ponto (terminar com o dado, não com legenda)
    x_ult, y_ult = df_plot['DATA'].iloc[-1], serie.iloc[-1]
    ax.annotate(f'{y_ult:.2g}', xy=(x_ult, y_ult), xytext=(8, 0), textcoords='offset points',
                va='center', fontsize=9.5, fontweight='bold', color=NESA_COLORS['primary'])

    # Escala honesta: mostra a folga até o VMP sem achatar a variação dos dados
    y_max = max(topo, float(serie.max())) * 1.12
    if isinstance(limite, tuple):
        y_min = min(float(serie.min()), base) * 0.95
    else:
        y_min = min(0.0, float(serie.min()))
    ax.set_ylim(y_min, y_max)

    if isinstance(limite, tuple):
        conforme = serie.between(limite[0], limite[1]).mean() * 100
    else:
        conforme = (serie <= limite).mean() * 100

    rotulo_y = "pH" if unidade == '' else f"Valor ({unidade})"
    sub = f"{conforme:.0f}% das {len(serie)} amostras dentro do limite CONAMA 430/11   |   P95 = {p95:.2g} {unidade}".strip()
    titulo_storytelling(
        ax,
        f"{parametro.replace('_', ' ').title()}  -  {ete}",
        sub,
        "Fonte: Monitoramento de Efluentes Sanitários NESA   |   Limite: Res. CONAMA 430/2011, Art. 16",
    )
    ax.set_ylabel(rotulo_y)
    formatar_meses_pt(ax)
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

    nome_arquivo = f"Plot_{parametro}_{ete.replace(' ', '_')}.png"
    fig.savefig(dir_out / nome_arquivo)
    plt.close(fig)

# --- 4. EXECUÇÃO DO PIPELINE ---
if __name__ == "__main__":
    try:
        df = pd.read_excel(FILE_INPUT)
        col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]

        df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
        df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

        relatorio_geral, relatorio_kw = [], []
        print("[*] Iniciando processamento estatístico completo...")

        for param in VMP_CONAMA.keys():
            if param in df.columns:
                print(f"    -> Analisando: {param}")
                relatorio_geral.extend(analisar_parametro(df, param, VMP_CONAMA[param], col_ete))
                resultado_kw = teste_kruskal_wallis_geral(df, param, col_ete)
                if resultado_kw:
                    relatorio_kw.append(resultado_kw)
                for ete in df[col_ete].unique():
                    plotar_serie_temporal(df, param, ete, col_ete, DIR_PRODUTOS)

        df_final = pd.DataFrame(relatorio_geral)
        df_kw = pd.DataFrame(relatorio_kw)

        caminho_excel = DIR_PRODUTOS / "Relatorio_Analitico_Mestre.xlsx"
        with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
            df_final.to_excel(writer, sheet_name="Estatistica_e_Tendencia", index=False)
            if not df_kw.empty:
                df_kw.to_excel(writer, sheet_name="Comparativo_Kruskal_Wallis", index=False)

        print(f"\n[+] SUCESSO! Relatório Mestre salvo em: {caminho_excel}")
    except Exception as e:
        print(f"[!] Erro crítico: {e}")


[*] Iniciando processamento estatístico completo...
    -> Analisando: DEMANDA_BIOQUIMICA_DE_OXIGENIO
    -> Analisando: PH
    -> Analisando: TEMPERATURA_DO_EFLUENTE
    -> Analisando: MATERIAIS_SEDIMENTAVEIS
    -> Analisando: OLEOS_E_GRAXAS
    -> Analisando: ARSENIO_TOTAL
    -> Analisando: CADMIO_TOTAL
    -> Analisando: CHUMBO_TOTAL
    -> Analisando: CIANETO_TOTAL
    -> Analisando: CROMO_HEXAVALENTE
    -> Analisando: CROMO_TRIVALENTE
    -> Analisando: FERRO_DISSOLVIDO
    -> Analisando: MANGANES_DISSOLVIDO
    -> Analisando: MERCURIO_TOTAL
    -> Analisando: SULFETO_TOTAL
    -> Analisando: BENZENO
    -> Analisando: CLOROFORMIO
    -> Analisando: ETILBENZENO
    -> Analisando: FENOIS_TOTAIS
    -> Analisando: TETRACLORETO_DE_CARBONO
    -> Analisando: TRICLOROETENO
    -> Analisando: TOLUENO
    -> Analisando: XILENO_TOTAL

[+] SUCESSO! Relatório Mestre salvo em: C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos\Relatorio_Analitico_Mestre.xlsx


In [2]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  19-04-2026  |  Revisado: 06-06-2026 (v2)
# Uso:     Módulo QA/QC: tabela + mapa de calor (storytelling) quantificando
#          os dados censurados (<LQ) por Parâmetro e por ETE.
# =============================================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", message="invalid value encountered")

# --- 1. CONFIGURAÇÕES ---
RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {'primary': '#009CA7', 'text': '#404041', 'muted': '#9aa0a6'}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# --- 2. EXECUÇÃO ---
try:
    print("[*] Lendo a base de dados processada para auditoria de QA/QC...")
    df = pd.read_excel(FILE_INPUT)

    col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
    df[col_ete] = df[col_ete].astype(str).str.upper().str.strip()
    df[col_ete] = df[col_ete].replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'})

    colunas_metadados = df.columns[:4].tolist()
    parametros = [
        c for c in df.columns
        if c not in colunas_metadados
        and not c.endswith('_IMPUTADO_LQ') and not c.endswith('_STATUS_VMP')
    ]

    dados_qc = []
    print("[*] Contabilizando matriz de dados censurados...")
    for param in parametros:
        col_flag = param + "_IMPUTADO_LQ"
        has_flag = col_flag in df.columns
        for ete in df[col_ete].unique():
            df_ete = df[df[col_ete] == ete]
            serie_param = df_ete[param].dropna()
            n_total = len(serie_param)
            if n_total > 0:
                if has_flag:
                    n_censurado = df_ete.loc[serie_param.index, col_flag].fillna(False).sum()
                else:
                    n_censurado = 0
                dados_qc.append({
                    'ETE': ete,
                    'Parâmetro': param.replace('_', ' '),
                    'Total de Amostras (N)': int(n_total),
                    'Dados Censurados (<LQ)': int(n_censurado),
                    '% Censurado': round((n_censurado / n_total) * 100, 1),
                })

    df_qc = pd.DataFrame(dados_qc)

    caminho_excel = DIR_PRODUTOS / "Relatorio_QAQC_Dados_Censurados.xlsx"
    df_qc.to_excel(caminho_excel, index=False)

    # --- MAPA DE CALOR (storytelling) ---
    df_qc_filtrado = df_qc[df_qc['% Censurado'] > 0]
    if not df_qc_filtrado.empty:
        pivot_qc = df_qc_filtrado.pivot(index='Parâmetro', columns='ETE', values='% Censurado').fillna(0)
        # Ordena por incidência média (linhas mais críticas no topo)
        ordem = pivot_qc.mean(axis=1).sort_values(ascending=False).index
        pivot_qc = pivot_qc.loc[ordem]

        fig, ax = plt.subplots(figsize=(9, max(6, len(pivot_qc) * 0.42)))
        sns.heatmap(
            pivot_qc, annot=True, fmt='.0f', cmap='YlGnBu', vmin=0, vmax=100,
            linewidths=1.2, linecolor='white',
            annot_kws={'fontsize': 9, 'fontweight': 'bold'},
            cbar_kws={'label': '% de amostras censuradas (<LQ)', 'shrink': 0.55}, ax=ax,
        )
        ax.set_xlabel(''); ax.set_ylabel('')
        ax.tick_params(length=0)
        plt.setp(ax.get_xticklabels(), fontweight='bold')

        ax.text(0.0, 1.06, "Matriz de Dados Censurados (<LQ) por ETE",
                transform=ax.transAxes, fontsize=14, fontweight='bold',
                color='#2b2b2b', va='bottom', ha='left')
        ax.text(0.0, 1.015,
                "Células mais escuras = maior fração de amostras abaixo do limite de quantificação (não detectado)",
                transform=ax.transAxes, fontsize=9.5, color=NESA_COLORS['muted'],
                va='bottom', ha='left')

        caminho_fig = DIR_PRODUTOS / "Heatmap_QAQC_Censurados.png"
        fig.savefig(caminho_fig)
        plt.close(fig)

    print(f"\n[+] SUCESSO! Produtos gerados em: {DIR_PRODUTOS}")
    print("  -> Tabela: Relatorio_QAQC_Dados_Censurados.xlsx")
    if not df_qc_filtrado.empty:
        print("  -> Gráfico: Heatmap_QAQC_Censurados.png")

    print("\n=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===")
    print(df_qc.sort_values(by='% Censurado', ascending=False).head(10).to_string(index=False))

except Exception as e:
    print(f"[!] Erro no processamento de QA/QC: {e}")


[*] Lendo a base de dados processada para auditoria de QA/QC...
[*] Contabilizando matriz de dados censurados...

[+] SUCESSO! Produtos gerados em: C:\Users\thiago\Documents\estudos_python\etes_nesa\Efluentes_Sanitários\produtos
  -> Tabela: Relatorio_QAQC_Dados_Censurados.xlsx
  -> Gráfico: Heatmap_QAQC_Censurados.png

=== TOP 10 MAIORES INCIDÊNCIAS DE CENSURA (<LQ) ===
         ETE                                Parâmetro  Total de Amostras (N)  Dados Censurados (<LQ)  % Censurado
      ETE 01                            ARSENIO TOTAL                     26                      26        100.0
      ETE PM                            ARSENIO TOTAL                     20                      20        100.0
ETE COMPACTA                         CROMO TRIVALENTE                     24                      24        100.0
      ETE PM                         CROMO TRIVALENTE                     20                      20        100.0
      ETE 01 DICLOROETENO SOMATORIA DE 1112CIS12TRANS   

In [3]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)
# Criado:  01-05-2026  |  Revisado: 06-06-2026 (v2.1)
# Uso:     Dossiê Visual Segregado por UHE (Belo Monte e Pimental).
# v2:      - P95 (DBO/NH3/Fósforo) AUTOMATIZADO a partir da base processada.
# v2.1:    - P95 SEGMENTADO por UHE: BM usa ETE 01+02; PM usa ETE PM+Compacta.
#          - Gráficos de impacto: escala FIXA mantida; subtítulo declara o
#            valor real e quantas vezes está abaixo da detecção (honesto).
#          - Eixo de datas em PORTUGUÊS (formatador independente de locale).
# =============================================================================

import datetime
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

# --- 1. CONFIGURAÇÕES E PASTAS ---
RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_VAZAO = DIR_BASE / "vazão turbinada.xlsx"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {
    'rio': '#B9F8FF', 'ete_barras': '#009CA7', 'linha_vmp': '#F25A3C',
    'dbo': '#009CA7', 'nh3': '#52BD8B', 'fosforo': '#FEBD2A',
    'lab': '#404041', 'muted': '#9aa0a6',
}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Calibri', 'Arial', 'DejaVu Sans'],
    'axes.axisbelow': True, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})

# Meses em portugues (independe de locale do sistema operacional)
MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def formatar_meses_pt(ax):
    """Eixo X de datas com abreviações de mês em PT-BR (ex.: Mai/25, Out/25)."""
    def _fmt(x, pos=None):
        d = mdates.num2date(x)
        return f"{MESES_PT[d.month - 1]}/{d:%y}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))

def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    ax.set_title('')
    ax.text(0.0, 1.12, titulo, transform=ax.transAxes, fontsize=15,
            fontweight='bold', color='#2b2b2b', va='bottom', ha='left')
    if subtitulo:
        ax.text(0.0, 1.035, subtitulo, transform=ax.transAxes, fontsize=10.5,
                color=NESA_COLORS['muted'], va='bottom', ha='left')
    if fonte:
        ax.text(0.0, -0.32, fonte, transform=ax.transAxes, fontsize=8,
                color=NESA_COLORS['muted'], va='top', ha='left')

def limpar_eixos(ax):
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
    ax.tick_params(length=0)

# --- 2. MOTOR DE TEMPO (BLINDADO) ---
def extrair_data_segura(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(year=val.year, month=val.month, day=1)
    s = str(val).lower()
    meses_pt = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
    for i, mes in enumerate(meses_pt):
        if mes in s:
            anos = re.findall(r'\d{2,4}', s)
            if anos:
                ano = int(anos[-1])
                if ano < 100:
                    ano += 2000
                return pd.Timestamp(year=ano, month=i + 1, day=1)
    return pd.NaT

# --- P95 SEGMENTADO POR UHE (substitui os números hardcoded) ---
def calcular_p95_segmento(df, col_ete, etes_segmento, candidatos, fallback):
    """Deriva o P95 (pior cenário crônico) de um parâmetro usando APENAS as
    ETEs do segmento informado - coerente com o balanço de massa segregado.
    Procura a 1a coluna existente em 'candidatos'. Retorna (valor, origem)."""
    df_seg = df
    if col_ete is not None:
        df_seg = df[df[col_ete].isin(etes_segmento)]
    for nome in candidatos:
        if nome in df_seg.columns:
            serie = pd.to_numeric(df_seg[nome], errors='coerce').dropna()
            if len(serie) >= 3:
                return (round(float(np.percentile(serie, 95)), 2),
                        f"P95 de '{nome}' em {etes_segmento} (N={len(serie)})")
    return fallback, f"FALLBACK (sem dados p/ {candidatos} em {etes_segmento})"

# Mapeamento parametro -> possiveis nomes de coluna na base processada
PARAM_COLS = {
    'DBO':  ['DEMANDA_BIOQUIMICA_DE_OXIGENIO'],
    'NH3':  ['NITROGENIO_AMONIACAL_TOTAL', 'NITROGENIO_AMONIACAL', 'N_AMONIACAL', 'NITROGENIO_AMONIACAL_NH3'],
    'FOSF': ['FOSFORO_TOTAL', 'FOSFORO', 'P_TOTAL', 'FOSFORO_TOTAL_P'],
}
FALLBACKS = {'DBO': 88.78, 'NH3': 45.0, 'FOSF': 4.5}
SEGMENTOS = {'BM': ['ETE 01', 'ETE 02'], 'PM': ['ETE PM', 'ETE COMPACTA']}

# --- 3. DADOS DE LANCAMENTO (GRI - m3/mes, Jan/24 a Mar/26) ---
dados_gri = {
    'ETE_01': [651.4, 635.7, 503.4, 675.4, 639.4, 569.0, 566.6, 547.0, 455.1, 679.9, 581.8, 551.6, 324.8, 169.6, 99.2, 155.2, 129.6, 110.4, 137.6, 120.0, 146.4, 160.0, 144.8, 92.0, 107.2, 111.2, 145.6],
    'ETE_02': [597.4, 525.1, 415.0, 535.8, 579.8, 469.4, 534.2, 438.6, 330.9, 581.3, 405.8, 387.2, 2414.4, 1060.8, 959.2, 1325.6, 1130.4, 640.8, 885.6, 580.0, 664.8, 688.0, 585.6, 648.8, 602.4, 502.4, 623.76],
    'ETE_PM': [87.0, 198.4, 206.1, 185.6, 215.0, 245.1, 294.4, 258.4, 291.0, 338.5, 308.6, 314.5, 281.6, 146.4, 133.76, 137.92, 142.56, 141.12, 126.72, 104.16, 104.48, 0.0, 109.92, 0.0, 0.0, 57.6, 65.92],
    'ETE_COMPACTA': [21.8, 49.6, 51.5, 46.4, 50.6, 52.5, 52.8, 52.0, 45.4, 52.3, 46.2, 46.7, 41.6, 16.0, 18.24, 22.08, 23.04, 20.48, 18.88, 15.84, 19.76, 25.12, 19.68, 19.04, 14.72, 14.4, 16.48],
}
df_gri = pd.DataFrame(dados_gri)
df_gri['DATA_DT'] = pd.date_range(start='2024-01-01', periods=len(df_gri), freq='MS')

LIMITE_INICIO = pd.Timestamp('2023-12-15')
LIMITE_FIM = pd.Timestamp('2026-03-15')

try:
    # --- P95 SEGMENTADO derivado da base processada (rastreável) ---
    p95 = {}     # p95[('DBO','BM')] = valor
    origem = {}  # origem[('DBO','BM')] = texto de rastreabilidade
    if FILE_INPUT.exists():
        df_proc = pd.read_excel(FILE_INPUT)
        col_ete = next((c for c in df_proc.columns
                        if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()), None)
        if col_ete is not None:
            df_proc[col_ete] = (df_proc[col_ete].astype(str).str.upper().str.strip()
                                .replace({'ETE 1': 'ETE 01', 'ETE 2': 'ETE 02'}))
        for p, candidatos in PARAM_COLS.items():
            for seg, etes in SEGMENTOS.items():
                p95[(p, seg)], origem[(p, seg)] = calcular_p95_segmento(
                    df_proc, col_ete, etes, candidatos, FALLBACKS[p])
    else:
        for p in PARAM_COLS:
            for seg in SEGMENTOS:
                p95[(p, seg)], origem[(p, seg)] = FALLBACKS[p], "FALLBACK (base não encontrada)"

    print("[*] P95 SEGMENTADO por UHE (rastreabilidade):")
    for seg in SEGMENTOS:
        print(f"  --- Segmento {seg} ({' + '.join(SEGMENTOS[seg])}) ---")
        for p in PARAM_COLS:
            print(f"      {p:5s} = {p95[(p, seg)]:8.2f} mg/L  ->  {origem[(p, seg)]}")

    # --- Vazão + cruzamento ---
    df_vazao = pd.read_excel(FILE_VAZAO)
    df_vazao.columns = ['Mes_Original', 'Q_Pimental_m3s', 'Q_BeloMonte_m3s']
    df_vazao = df_vazao.dropna(subset=['Mes_Original'])
    df_vazao['DATA_DT'] = df_vazao['Mes_Original'].apply(extrair_data_segura)
    df_vazao = df_vazao.dropna(subset=['DATA_DT'])

    df_final = pd.merge(df_gri, df_vazao, on='DATA_DT', how='left')

    # --- BALANCO DE MASSA (P95 especifico de cada segmento) ---
    for seg, q_col, etes in [('BM', 'Q_BeloMonte_m3s', ['ETE_01', 'ETE_02']),
                             ('PM', 'Q_Pimental_m3s', ['ETE_PM', 'ETE_COMPACTA'])]:
        df_final[f'Vol_Rio_{seg}_m3mes'] = df_final[q_col] * 86400 * 30.44
        df_final[f'Vol_Efluente_{seg}'] = df_final[etes].sum(axis=1)
        df_final[f'Fator_Diluicao_{seg}'] = df_final[f'Vol_Rio_{seg}_m3mes'] / df_final[f'Vol_Efluente_{seg}']
        for nome in ['DBO', 'NH3', 'FOSF']:
            df_final[f'Adicao_{nome}_{seg}_mgL'] = (
                df_final[f'Vol_Efluente_{seg}'] * p95[(nome, seg)]) / df_final[f'Vol_Rio_{seg}_m3mes']

    print("\n" + "=" * 80)
    print("  DOSSIÊ VISUAL SEGREGADO POR UHE (v2.1 - P95 segmentado - Jan/24 a Mar/26)")
    print("=" * 80)

    # =====================================================================
    # GRÁFICO: PROPORÇÃO DE DILUIÇÃO (barras)
    # =====================================================================
    def gerar_grafico_diluicao(fator_col, uhe_nome, arquivo_nome):
        fig, ax = plt.subplots(figsize=(14, 6))
        limpar_eixos(ax)
        fator_milhoes = df_final[fator_col] / 1e6

        idx_min = fator_milhoes.idxmin() if fator_milhoes.notna().any() else None
        cores = [NESA_COLORS['linha_vmp'] if i == idx_min else NESA_COLORS['ete_barras']
                 for i in fator_milhoes.index]
        bars = ax.bar(df_final['DATA_DT'], fator_milhoes, width=22, color=cores, alpha=0.92)

        labels = [f'{v:.1f}' if pd.notna(v) else '' for v in fator_milhoes]
        ax.bar_label(bars, labels=labels, padding=4, fontsize=8.5, color='#404041')

        topo = fator_milhoes.max() * 1.18 if pd.notna(fator_milhoes.max()) else 10
        ax.set_ylim(0, topo)
        ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)
        ax.grid(axis='y', linestyle='--', alpha=0.35)
        ax.grid(axis='x', visible=False)

        min_fator = fator_milhoes.min()
        subt = ""
        if pd.notna(min_fator):
            x_min = df_final['DATA_DT'].iloc[df_final.index.get_loc(idx_min)]
            ax.annotate(f'Pior mês: 1 p/ {min_fator:.1f} M',
                        xy=(x_min, min_fator), xytext=(0, 28), textcoords='offset points',
                        ha='center', fontsize=9.5, fontweight='bold', color=NESA_COLORS['linha_vmp'],
                        arrowprops=dict(arrowstyle='->', color=NESA_COLORS['linha_vmp'], lw=1.6))
            subt = (f"Mesmo no pior mês histórico, cada 1 L de efluente é diluído em "
                    f"{min_fator:.1f} milhões de litros do Rio Xingu")

        titulo_storytelling(
            ax, f"Proporção Hidrodinâmica - UHE {uhe_nome}", subt,
            "Fonte: Volumes GRI 303-4 (NESA) x Vazão turbinada (ONS)   |   Vol.rio = Q x 86400 x 30,44",
        )
        ax.set_ylabel("Milhões de litros de rio por 1 L de efluente", fontweight='bold')
        formatar_meses_pt(ax)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arquivo_nome)
        plt.close(fig)

    # =====================================================================
    # GRÁFICO: IMPACTO (escala FIXA + subtítulo honesto)
    # =====================================================================
    def gerar_grafico_impacto(param_col, cor, nome_parametro, p95_seg, limite_lab,
                              nota_conama, limite_y, uhe_nome, arquivo_nome):
        fig, ax = plt.subplots(figsize=(12, 5.4))
        limpar_eixos(ax)

        mask = df_final[param_col].notna()
        ax.plot(df_final['DATA_DT'], df_final[param_col], color=cor, linewidth=3,
                label=f'Adição efetiva de {nome_parametro}', zorder=3)
        ax.fill_between(df_final['DATA_DT'][mask], 0, df_final[param_col][mask], color=cor, alpha=0.18)

        ax.axhline(limite_lab, color=NESA_COLORS['lab'], linestyle='-', linewidth=2,
                   label=f'Limite de detecção lab. ({limite_lab} mg/L)')

        # ESCALA FIXA mantida (formato consagrado)
        ax.set_ylim(0, limite_y)
        ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)

        ax.annotate(nota_conama,
                    xy=(pd.Timestamp('2025-01-01'), limite_y * 0.96),
                    xytext=(pd.Timestamp('2025-01-01'), limite_y * 0.82),
                    color=NESA_COLORS['linha_vmp'], fontweight='bold', fontsize=10.5, ha='center',
                    arrowprops=dict(facecolor=NESA_COLORS['linha_vmp'], arrowstyle='->', lw=2))

        # Subtítulo HONESTO: valor real + quantas vezes abaixo da detecção
        adic_max = df_final[param_col].max()
        if pd.notna(adic_max) and adic_max > 0:
            fator = limite_lab / adic_max
            subt = (f"P95 do efluente = {p95_seg:g} mg/L -> adição máxima de {adic_max:.5f} mg/L  "
                    f"({fator:,.0f}x abaixo do limite de detecção do laboratório)")
        else:
            subt = f"P95 do efluente = {p95_seg:g} mg/L -> adição efetiva próxima de zero após diluição"

        titulo_storytelling(
            ax, f"Modelagem de Risco - UHE {uhe_nome}: {nome_parametro}", subt,
            "Fonte: Balanço de massa NESA (P95 segmentado do efluente x diluição do Rio Xingu)   |   CONAMA 430/2011",
        )
        ax.set_ylabel("Concentração adicionada (mg/L)", fontweight='bold')
        ax.grid(axis='y', linestyle=':', alpha=0.5)
        ax.grid(axis='x', visible=False)
        formatar_meses_pt(ax)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arquivo_nome)
        plt.close(fig)

    # --- BELO MONTE ---
    print("[*] Gerando pacote de gráficos para UHE Belo Monte...")
    gerar_grafico_diluicao('Fator_Diluicao_BM', 'Belo Monte', "Produto1A_Diluicao_BeloMonte.png")
    gerar_grafico_impacto('Adicao_DBO_BM_mgL', NESA_COLORS['dbo'], 'DBO', p95[('DBO', 'BM')], 2.0,
                          'Limite CONAMA (120 mg/L) -> Fora de escala', 2.5, 'Belo Monte', "Produto2A_Impacto_DBO_BM.png")
    gerar_grafico_impacto('Adicao_NH3_BM_mgL', NESA_COLORS['nh3'], 'N. Amoniacal', p95[('NH3', 'BM')], 0.5,
                          'Limite CONAMA (20 mg/L) -> Fora de escala\n*Isento pelo Art. 21', 0.7, 'Belo Monte', "Produto3A_Impacto_NH3_BM.png")
    gerar_grafico_impacto('Adicao_FOSF_BM_mgL', NESA_COLORS['fosforo'], 'Fósforo Total', p95[('FOSF', 'BM')], 0.5,
                          'Risco de eutrofização -> Fora de escala', 0.7, 'Belo Monte', "Produto4A_Impacto_Fosforo_BM.png")

    # --- PIMENTAL ---
    print("[*] Gerando pacote de gráficos para UHE Pimental...")
    gerar_grafico_diluicao('Fator_Diluicao_PM', 'Pimental (TVR)', "Produto1B_Diluicao_Pimental.png")
    gerar_grafico_impacto('Adicao_DBO_PM_mgL', NESA_COLORS['dbo'], 'DBO', p95[('DBO', 'PM')], 2.0,
                          'Limite CONAMA (120 mg/L) -> Fora de escala', 2.5, 'Pimental', "Produto2B_Impacto_DBO_PM.png")
    gerar_grafico_impacto('Adicao_NH3_PM_mgL', NESA_COLORS['nh3'], 'N. Amoniacal', p95[('NH3', 'PM')], 0.5,
                          'Limite CONAMA (20 mg/L) -> Fora de escala\n*Isento pelo Art. 21', 0.7, 'Pimental', "Produto3B_Impacto_NH3_PM.png")
    gerar_grafico_impacto('Adicao_FOSF_PM_mgL', NESA_COLORS['fosforo'], 'Fósforo Total', p95[('FOSF', 'PM')], 0.5,
                          'Risco de eutrofização -> Fora de escala', 0.7, 'Pimental', "Produto4B_Impacto_Fosforo_PM.png")

    # --- PLANILHA MESTRA (inclui o P95 usado em cada segmento) ---
    df_final['Mes/Ano'] = df_final['DATA_DT'].dt.strftime('%m/%Y')
    df_final['P95_DBO_BM'] = p95[('DBO', 'BM')]
    df_final['P95_DBO_PM'] = p95[('DBO', 'PM')]
    colunas_export = [
        'Mes/Ano',
        'Q_BeloMonte_m3s', 'Vol_Rio_BM_m3mes', 'Vol_Efluente_BM', 'Fator_Diluicao_BM',
        'Adicao_DBO_BM_mgL', 'Adicao_NH3_BM_mgL', 'Adicao_FOSF_BM_mgL',
        'Q_Pimental_m3s', 'Vol_Rio_PM_m3mes', 'Vol_Efluente_PM', 'Fator_Diluicao_PM',
        'Adicao_DBO_PM_mgL', 'Adicao_NH3_PM_mgL', 'Adicao_FOSF_PM_mgL',
    ]
    df_final[colunas_export].to_excel(DIR_PRODUTOS / "Relatorio_Diluicao_Mestre_Segregado.xlsx", index=False)

    print("\n[*] Concluído! 8 gráficos segregados (P95 por UHE) salvos na pasta 'produtos'.")

except Exception as e:
    print(f"\n[!] ERRO CRÍTICO ENCONTRADO: {e}")


[*] P95 SEGMENTADO por UHE (rastreabilidade):
  --- Segmento BM (ETE 01 + ETE 02) ---
      DBO   =    86.12 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE 01', 'ETE 02'] (N=78)
      NH3   =   266.05 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE 01', 'ETE 02'] (N=78)
      FOSF  =     8.80 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE 01', 'ETE 02'] (N=78)
  --- Segmento PM (ETE PM + ETE COMPACTA) ---
      DBO   =   188.37 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      NH3   =   184.16 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      FOSF  =    27.83 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)

  DOSSIÊ VISUAL SEGREGADO POR UHE (v2.1 - P95 segmentado - Jan/24 a Mar/26)
[*] Gerando pacote de gráficos para UHE Belo Monte...
[*] Gerando pacote de gráficos para UHE Pimental...

[*] Concluído! 8 gráficos segregados (P95 por UHE) salvos na pasta 'produtos'.
